# shortest-vs-fewest

**What does it cost to skip a stop?**

A shortest route by distance and a shortest route by number of legs are two
different questions. This experiment asks both for the same pairs, on the same
frozen data, and looks at the gap between them.

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a different
answer.

## Setup

Only the first cell differs between Colab and a local checkout.

In [ ]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/<owner>/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit("install the package first -- see the comment above") from error

In [ ]:
from pathlib import Path

from flight_planner import BFS, Dijkstra
from flight_planner.experiments import Experiment

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
pairs = [tuple(pair) for pair in experiment.parameters['pairs']]
pairs

## The data

Opening the snapshot re-hashes every file against the manifest.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria)

catalog = experiment.catalog()
planner = catalog.planner()
print(f'{len(catalog.routes):,} routes / {len(catalog.airports):,} airports')

The snapshot was frozen at `airport_type=large_airport, country=US`, so the
narrowing is already done and this notebook does not narrow further. The
`criteria` printed above say so, and they are copied into the recorded result.

## The question

`find_shortest_route` takes the algorithm as its third argument. `Dijkstra`
minimises the summed edge weight -- kilometres. `BFS` minimises the number of
edges -- legs.

One catch worth knowing before reading any number below: **the first value BFS
returns is a hop count, not a distance.** It ignores `edge.weight` entirely, so
it cannot report kilometres. Sum the legs yourself for that.

In [ ]:
def kilometres(legs):
    """Total distance of a list of legs."""
    return sum(leg.distance_km for leg in legs)


def route_via(origin, legs):
    """Render a route as 'SFO->DEN->BOS'."""
    return '->'.join([origin] + [leg.destination.iata_code for leg in legs])


for origin, destination in pairs:
    by_km, km_legs = planner.find_shortest_route(origin, destination, Dijkstra())
    hops, hop_legs = planner.find_shortest_route(origin, destination, BFS())

    print(f'{origin} -> {destination}')
    print(f'  Dijkstra: {by_km:9,.0f} km  {len(km_legs)} leg(s)  {route_via(origin, km_legs)}')
    print(f'  BFS     : {kilometres(hop_legs):9,.0f} km  {hops:.0f} leg(s)  {route_via(origin, hop_legs)}')

`HNL -> BDL` does not read the way the others do. Dijkstra finds three legs for
8,072 km; BFS finds two for 8,616 km. Taken at face value that says skipping a
stop costs 544 km -- far more than the 155 km it costs on `BOI -> CHS`.

It is worth checking that before believing it.

## Is BFS's route the best two-leg route?

BFS returns the *first* minimum-hop path it finds. Nothing in it compares the
paths of equal length, because it never looks at distance at all. So its answer
is *a* shortest-by-legs route, not the shortest one among them.

The catalog can answer the question BFS cannot: of every two-leg route between
this pair, which is shortest?

In [ ]:
def best_two_leg(origin, destination):
    """Shortest two-leg route between a pair, by distance.

    Every route out of `origin` paired with every route from there into
    `destination` -- small enough to enumerate directly on this network.
    """
    candidates = [
        (first.distance_km + second.distance_km, first.destination.iata_code)
        for first in catalog.routes_from(origin)
        for second in catalog.routes_from(first.destination.iata_code)
        if second.destination.iata_code == destination
    ]
    return min(candidates) if candidates else None


for origin, destination in [('BOI', 'CHS'), ('HNL', 'BDL')]:
    _, hop_legs = planner.find_shortest_route(origin, destination, BFS())
    best_km, hub = best_two_leg(origin, destination)

    print(f'{origin} -> {destination}')
    print(f'  BFS returned    : {kilometres(hop_legs):9,.0f} km  via {route_via(origin, hop_legs)}')
    print(f'  best two-leg    : {best_km:9,.0f} km  via {hub}')
    print(f'  BFS overshoot   : {kilometres(hop_legs) - best_km:9,.0f} km')

There it is. On `BOI -> CHS` BFS happened to land on the best two-leg route. On
`HNL -> BDL` it returned one 540 km worse than necessary, and that overshoot --
not the cost of skipping a stop -- is what made the pair look expensive.

The real cost of dropping to two legs on `HNL -> BDL` is about 4 km.

Neither algorithm is wrong. They answer different questions, and BFS makes no
promise at all about *which* of the equally-short-by-legs routes it hands
back.

## The answer

Recorded next to the data that produced it. Per pair: what each algorithm
found, the best route at BFS's leg count, and the two differences that matter
-- how far BFS overshot, and what a stop is actually worth.

In [ ]:
rows = []
for origin, destination in pairs:
    by_km, km_legs = planner.find_shortest_route(origin, destination, Dijkstra())
    hops, hop_legs = planner.find_shortest_route(origin, destination, BFS())

    row = {
        'pair': f'{origin}-{destination}',
        'dijkstra_km': by_km,
        'dijkstra_legs': len(km_legs),
        'dijkstra_route': route_via(origin, km_legs),
        'bfs_legs': int(hops),
        'bfs_km': kilometres(hop_legs),
        'bfs_route': route_via(origin, hop_legs),
    }

    if len(hop_legs) == 2:
        best_km, hub = best_two_leg(origin, destination)
        row['best_at_bfs_legs_km'] = best_km
        row['best_at_bfs_legs_via'] = hub
        row['bfs_overshoot_km'] = round(row['bfs_km'] - best_km, 3)
        row['cost_of_one_fewer_leg_km'] = round(best_km - by_km, 3)

    rows.append(row)

experiment.record({'pairs': rows}, catalog=catalog)